In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from collections import Counter
import gradio as gr

# ========== 0. 固定 random seed ==========
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

os.makedirs('./data', exist_ok=True)  # 確保data資料夾存在

# ========== 1. 讀取與合併資料 ==========
def load_and_merge_data():
    winners = pd.read_csv('./data/winners.csv')
    drivers = pd.read_csv('./data/drivers_updated.csv')
    teams = pd.read_csv('./data/teams_updated.csv')
    winners['year'] = pd.to_datetime(winners['Date']).dt.year.astype(int)
    for df_ in [winners, drivers, teams]:
        for col in df_.columns:
            df_[col] = df_[col].astype(str).str.strip()
    winners['year'] = winners['year'].astype(int)
    drivers['year'] = drivers['year'].astype(int)
    teams['year'] = teams['year'].astype(int)
    df = winners.merge(
        drivers[['Driver', 'Nationality', 'Car', 'year']],
        left_on=['Winner', 'Car', 'year'],
        right_on=['Driver', 'Car', 'year'],
        how='left'
    )
    df = df.rename(columns={'Car': 'Team'})
    df = df[['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']]
    for col in ['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']:
        df[col] = df[col].astype(str).str.strip()
    df = df.dropna(subset=['year', 'Grand Prix', 'Winner', 'Team', 'Driver', 'Nationality']).reset_index(drop=True)
    return df, drivers

df, drivers = load_and_merge_data()

# ========== 2. 只保留出現兩次以上的冠軍 ==========
vc = df['Winner'].value_counts()
multi_winner = vc[vc >= 2].index
df = df[df['Winner'].isin(multi_winner)].reset_index(drop=True)

# ========== 3. 分組，避免資料洩漏 ==========
df['race_id'] = df['year'].astype(str) + "_" + df['Grand Prix']
unique_race_ids = df['race_id'].unique()
train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=SEED)
train_df = df[df['race_id'].isin(train_ids)].reset_index(drop=True)
test_df = df[df['race_id'].isin(test_ids)].reset_index(drop=True)

# ========== 4. Encoder/Scaler fit only on train ==========
cat_cols = ['Grand Prix', 'Team', 'Driver', 'Nationality']
num_cols = ['year']
target_col = 'Winner'

def safe_transform(le, x):
    try:
        if x in le.classes_:
            return le.transform([x])[0]
        else:
            return 0
    except:
        return 0

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(train_df[col])
    train_df[col] = le.transform(train_df[col])
    test_df[col] = test_df[col].astype(str).apply(lambda x: safe_transform(le, x))
    label_encoders[col] = le

scaler = StandardScaler()
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])

# Winner label encoding可全fit
target_le = LabelEncoder().fit(df[target_col])
train_df['target'] = target_le.transform(train_df[target_col])
test_df['target'] = target_le.transform(test_df[target_col])

# ========== 5. Dataset & DataLoader ==========
class F1Dataset(Dataset):
    def __init__(self, df):
        self.cat = df[cat_cols].values
        self.num = df[num_cols].values
        self.y = df['target'].values
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        cat = np.where(self.cat[idx] < 0, 0, self.cat[idx])
        return torch.tensor(cat, dtype=torch.long), \
               torch.tensor(self.num[idx], dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.long)

batch_size = 64
trainset = F1Dataset(train_df)
testset = F1Dataset(test_df)

class_sample_count = np.array([len(np.where(train_df['target']==t)[0]) for t in np.unique(train_df['target'])])
weight = 1. / class_sample_count
label_counts = train_df['target'].value_counts().to_dict()
samples_weight = train_df['target'].map(lambda t: 1. / label_counts[t]).values
samples_weight = torch.from_numpy(samples_weight).float()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))
trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler)
testloader = DataLoader(testset, batch_size=batch_size)

# ========== 6. 強化版 DNN+Embedding 模型 ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num, emb_dim=16, hidden_dim=256, num_classes=20):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_num)
        self.fc1 = nn.Linear(len(cat_cols)*emb_dim + num_num, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.fc3 = nn.Linear(hidden_dim//2, num_classes)
    def forward(self, x_cat, x_num):
        x_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        x_num = self.bn_num(x_num)
        x = torch.cat([x_emb, x_num], dim=1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

cat_dims = [len(le.classes_) for le in label_encoders.values()]
num_classes = len(target_le.classes_)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = F1DNN(cat_dims, len(num_cols), emb_dim=16, hidden_dim=256, num_classes=num_classes).to(device)

# ========== 7. 訓練（記錄loss曲線）==========
def train_model(model, trainloader, testloader, n_epoch=100, lr=0.0003, patience=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_loss = np.inf
    no_improve = 0
    train_losses = []
    test_losses = []
    for epoch in range(n_epoch):
        model.train()
        total_loss = 0
        for x_cat, x_num, y in trainloader:
            x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
            logits = model(x_cat, x_num)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(trainloader)
        train_losses.append(avg_loss)
        # test loss
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y in testloader:
                x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
                logits = model(x_cat, x_num)
                loss = criterion(logits, y)
                test_loss += loss.item()
        avg_test_loss = test_loss / len(testloader)
        test_losses.append(avg_test_loss)
        print(f"Epoch {epoch+1}/{n_epoch} | Train Loss: {avg_loss:.4f} | Test Loss: {avg_test_loss:.4f}")
        if avg_loss < best_loss:
            best_loss = avg_loss
            no_improve = 0
            torch.save(model.state_dict(), './data/f1_dnn_embedding.pth')
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping triggered!")
                break
    # 畫 loss 曲線
    plt.figure(figsize=(7,5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train vs. Test Loss Curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig('./data/loss_curve.png')
    plt.close()
    return train_losses, test_losses

# ========== 8. Top-N (預設10) 冠軍混淆矩陣 ==========
def plot_topN_confusion_matrix(y_true, y_pred, target_names, topN=10, save_path='./data/confusion_matrix_topN.png'):
    # 找出最常出現的前 N 名
    top_classes = [x[0] for x in Counter(y_true).most_common(topN)]
    mask = np.isin(y_true, top_classes)
    filtered_y_true = [y for y, m in zip(y_true, mask) if m]
    filtered_y_pred = [y for y, m in zip(y_pred, mask) if m]
    filtered_labels = [target_names[i] for i in top_classes]
    cm = confusion_matrix(filtered_y_true, filtered_y_pred, labels=top_classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=filtered_labels)
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(xticks_rotation=45, ax=ax, cmap='Blues', colorbar=True)
    plt.title(f"Top {topN} Confusion Matrix (Test Set)")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ========== 9. 驗證 ==========
def eval_model(model, testloader, topN=10):
    model.eval()
    all_y, all_pred = [], []
    with torch.no_grad():
        for x_cat, x_num, y in testloader:
            x_cat, x_num = x_cat.to(device), x_num.to(device)
            logits = model(x_cat, x_num)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_pred.extend(preds)
            all_y.extend(y.numpy())
    acc = accuracy_score(all_y, all_pred)
    print(f"Test Accuracy: {acc:.4f}")
    labels = np.unique(np.concatenate([all_y, all_pred]))
    target_names = target_le.inverse_transform(labels)
    print(classification_report(all_y, all_pred, labels=labels, target_names=target_names))
    # 畫 Top-N 混淆矩陣
    plot_topN_confusion_matrix(all_y, all_pred, target_le.classes_, topN=topN)
    return acc

# ========== 10. 訓練＆驗證 ==========
train_losses, test_losses = train_model(model, trainloader, testloader, n_epoch=100, lr=0.0003, patience=10)
model.load_state_dict(torch.load('./data/f1_dnn_embedding.pth'))
eval_model(model, testloader, topN=10)

# ========== 11. 全年份/場地/車手對照 ==========
drivers['year'] = drivers['year'].astype(int)
drivers_year_dict = {}
teams_year_dict = {}
driver_team_nat_dict = {}
for y in sorted(drivers['year'].unique()):
    drivers_this_year = drivers[drivers['year']==y]
    driver_names = sorted(drivers_this_year['Driver'].dropna().astype(str).unique())
    drivers_year_dict[y] = driver_names
    teams_year_dict[y] = sorted(drivers_this_year['Car'].dropna().astype(str).unique())
    driver_team_nat_dict[y] = {}
    for _, row in drivers_this_year.iterrows():
        driver_team_nat_dict[y][row['Driver']] = (row['Car'], row['Nationality'])

# ========== 12. 年份與場地選單 ==========
all_years = sorted(df['year'].astype(int).unique())
all_grandprix = sorted(df['Grand Prix'].unique())

# ========== 13. Gradio預測前五名 ==========
def gradio_predict(year, grand_prix):
    year = int(year)
    if year not in drivers_year_dict:
        return "該年份查無參賽車手資料！"
    driver_names = drivers_year_dict[year]
    driver_team_nat = driver_team_nat_dict[year]
    all_results = []
    for driver in driver_names:
        team, nationality = driver_team_nat.get(driver, ("", ""))
        input_dict = {
            'year': float(year),
            'Grand Prix': grand_prix,
            'Team': team,
            'Driver': driver,
            'Nationality': nationality
        }
        x_cat = []
        for col in cat_cols:
            le = label_encoders[col]
            idx = le.transform([input_dict[col]])[0] if input_dict[col] in le.classes_ else 0
            x_cat.append(idx)
        x_cat = torch.tensor([x_cat], dtype=torch.long)
        x_num = torch.tensor([[input_dict['year']]], dtype=torch.float32)
        x_num = torch.tensor(scaler.transform(x_num), dtype=torch.float32)
        model.eval()
        with torch.no_grad():
            logits = model(x_cat.to(device), x_num.to(device))
            prob = torch.softmax(logits, dim=1).cpu().numpy().flatten()
        try:
            win_idx = target_le.transform([driver])[0]
            prob_value = prob[win_idx]
            all_results.append((driver, team, prob_value))
        except Exception as e:
            continue
    if not all_results:
        return "該年參賽車手無法進行預測"
    all_results.sort(key=lambda x: x[2], reverse=True)
    text = f"年份：{year} 場地：{grand_prix}\n\n預測冠軍機率排行（前5名）：\n\n"
    for i, (driver, team, prob_value) in enumerate(all_results[:5], 1):
        text += f"{i}. {driver}（{team}）：{prob_value:.2%}\n"
    if all_results[0][2] > 0.98:
        text += "\n⚠️ 機率極高可能代表資料極不平衡或模型未見過此組合，僅供參考。\n"
    return text

# ========== 14. Gradio UI ==========
with gr.Blocks() as demo:
    gr.Markdown("## F1 冠軍預測互動系統（含訓練/測試 Loss 曲線與 Top-10 混淆矩陣）")
    with gr.Tab("冠軍預測"):
        year_input = gr.Dropdown(label="年份", choices=all_years, value=all_years[-1])
        grand_prix = gr.Dropdown(label="Grand Prix 場地", choices=all_grandprix, value=all_grandprix[0])
        predict_btn = gr.Button("預測該場冠軍")
        output = gr.Textbox(label="預測結果")
        predict_btn.click(
            gradio_predict, 
            inputs=[year_input, grand_prix],
            outputs=output
        )
    with gr.Tab("訓練/測試 Loss 曲線"):
        gr.Markdown("以下為訓練過程 loss 曲線（訓練結束自動生成）：")
        gr.Image(value="./data/loss_curve.png", label="Loss Curve")
    with gr.Tab("Top-10 混淆矩陣（Test）"):
        gr.Markdown("以下為測試集 Top-10 頻率冠軍混淆矩陣（訓練結束自動生成）：")
        gr.Image(value="./data/confusion_matrix_topN.png", label="Confusion Matrix")
demo.launch()



c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/100 | Train Loss: 4.3574 | Test Loss: 4.3403
Epoch 2/100 | Train Loss: 4.2912 | Test Loss: 4.2801
Epoch 3/100 | Train Loss: 4.2149 | Test Loss: 4.2122
Epoch 4/100 | Train Loss: 4.1175 | Test Loss: 4.1141
Epoch 5/100 | Train Loss: 4.0043 | Test Loss: 3.9871
Epoch 6/100 | Train Loss: 3.8561 | Test Loss: 3.8137
Epoch 7/100 | Train Loss: 3.6414 | Test Loss: 3.6271
Epoch 8/100 | Train Loss: 3.4620 | Test Loss: 3.3901
Epoch 9/100 | Train Loss: 3.1809 | Test Loss: 3.0940
Epoch 10/100 | Train Loss: 2.9752 | Test Loss: 2.7912
Epoch 11/100 | Train Loss: 2.6270 | Test Loss: 2.4685
Epoch 12/100 | Train Loss: 2.3942 | Test Loss: 2.1974
Epoch 13/100 | Train Loss: 2.0589 | Test Loss: 1.9837
Epoch 14/100 | Train Loss: 1.8326 | Test Loss: 1.7574
Epoch 15/100 | Train Loss: 1.6494 | Test Loss: 1.5626
Epoch 16/100 | Train Loss: 1.4707 | Test Loss: 1.4211
Epoch 17/100 | Train Loss: 1.3658 | Test Loss: 1.3118
Epoch 18/100 | Train Loss: 1.2831 | Test Loss: 1.2215
Epoch 19/100 | Train Loss: 1.1786 | T

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f